In [1]:
import sys
import torch
import random
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Add project root to path for modular imports
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Required library:
# !pip install tf-keras

# 2. Import custom modules to handle the data fetching and processing
from src.data_loader import DataOrchestrator
from src.data_processor import FeatureProcessor

# 3. Enable autoreload to catch changes in .py files instantly
%load_ext autoreload
%autoreload 2

print("🚀 Environment Ready.")


🚀 Environment Ready.


In [2]:
# --- 1. Stocks Selection ---

# You can change this list to any N stocks you want to experiment with
STOCKS_LIST = [
    "AAPL", "ABBV", "ADBE", "AMD", "AMT", "AMZN", "AVGO", "BAC", "BHP", "BLK",
    "BP", "COP", "COST", "CVX", "DLR", "EQIX", "FCX", "GOOGL", "GS", "INTC",
    "JNJ", "JPM", "LIN", "LLY", "META", "MRK", "MS", "NEM", "NFLX", "NVDA",
    "O", "PEP", "PFE", "PLD", "QCOM", "RIO", "SCCO", "SCHW", "SHW", "SLB",
    "SPG", "TMO", "TSLA", "TTE", "UNH", "WELL", "WFC", "XOM"
]

# --- 2. Hyperparameters ---

CONFIG = {
    'M': 55,                # Lookback window
    'T': 5,                 # Prediction horizon
    'ATR_MULTIPLIER': 1.5,  # For labeling threshold
    'NEWS_THRESHOLD': 10,   # How many top stocks to include in the fused dataset
}

# --- 3. API Keys & Ranges ---
API_KEYS = {
    'tiingo': 'YOUR_API',
    'alpha_vantage': 'YOUR_API',
    'eodhd': 'YOUR_API',
}

DATES = {
    'start': "2019-01-01", # Buffer for technical indicators
    'end': "2025-12-31"
}

print(f"⚙️ Configured for {len(STOCKS_LIST)} stocks with M={CONFIG['M']}, T={CONFIG['T']}.")

⚙️ Configured for 2 stocks with M=55, T=5.


Step 1: Sync Raw Data - Skip this cell if you already have the raw data

In [7]:
print("📡 Step 1: Syncing Raw Data...")
loader = DataOrchestrator(API_KEYS, DATES)
loader.sync_all(STOCKS_LIST)

📡 Step 1: Syncing Raw Data...


Syncing Raw Data:   0%|          | 0/2 [00:00<?, ?it/s]

prices data for OKLO exists for 2024-12-02 - 2025-12-31, skipping
earnings data for OKLO exists for 2025-03-24 - 2025-11-13, skipping
Fetching OKLO news
prices data for RGTI exists for 2024-12-02 - 2025-12-31, skipping
earnings data for RGTI exists for 2025-03-05 - 2025-11-10, skipping
Fetching RGTI news

🏁 Synchronization Complete. Data updated for 2 tickers.


Step 2: Process Datasets

In [4]:
# --- Technical datasets ---

print("\n👨‍🍳 Step 2: Processing Datasets...")
generator = FeatureProcessor()

# 1. Generate the Technical + ER
generator.generate_tech_er_dataset(
    stocks=STOCKS_LIST,
    dates={'start': DATES['start'], 'end': DATES['end']},
    M=CONFIG['M'],
    T=CONFIG['T'],
    multiplier=CONFIG['ATR_MULTIPLIER']
)


👨‍🍳 Step 2: Processing Datasets...


Processing Technicals:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Tech+ER Cube Saved: 398 samples.


In [4]:
# --- Fused datasets, including news - skip if you don't have news data

# 2. Generate the Technical + ER + News Dataset
generator.generate_tech_er_news_dataset(
    stocks=STOCKS_LIST,
    dates={'start': DATES['start'], 'end': DATES['end']},
    M=CONFIG['M'],
    T=CONFIG['T'],
    multiplier=CONFIG['ATR_MULTIPLIER'],
    news_threshold=CONFIG['NEWS_THRESHOLD']
)


👨‍🍳 Step 2: Processing Datasets...


Processing Technicals: 100%|██████████| 1/1 [00:00<00:00, 26.22it/s]

✅ Tech+ER Cube Saved: 0 samples.



C:\Users\omrym\anaconda3\envs\deep_learn\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\omrym\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP 

KeyError: 'date'